[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abraham1439/Rag_Reglamento_Biblioteca/blob/main/notebook/asistente_biblioteca_duoc.ipynb)

# Asistente Virtual RAG de Biblioteca Duoc UC

Este notebook reúne todo el código del pipeline RAG del proyecto **Biblioteca Duoc UC** en un solo archivo ejecutable directamente en **Google Colab**.

### Arquitectura del Notebook:
1. **Instalación de Dependencias**: `openai`, `langchain`, `faiss-cpu`, etc.
2. **Configuración y Claves API**: Groq (LLM) y Mistral (Embeddings).
3. **Validacion de los documentos existentes** : Validar que los documentos existen.
4. **Carga y Fragmentación (Ingesta) y Índice Vectorial (FAISS)**: Creación de documentos del reglamento y chunking, Generación de embeddings e indexación.
5. **Prompt Engineering**: Estrategias Zero-Shot, Few-Shot y Chain-of-Thought.
6. **“Agente RAG y filtro de recuperación**: Lógica de respuesta y derivación.
7. **Ejemplos de consultas interactivas**: Se prueba el modelo con preguntas que van al caso y preguntas que no
8. **Evalucion con casos dentro y fuera de alcance**:  Pruebas con consultas dentro y fuera de alcance.

## Clonar el repositorio desde colab (Solo ejecutar en caso de que el proyecto se ejecute desde colab)

In [ ]:
%cd /content
!git clone https://github.com/Abraham1439/Rag_Reglamento_Biblioteca.git
%cd /content/Rag_Reglamento_Biblioteca

## 1. Instalación de Dependencias

In [1]:
!pip install -q openai langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración del Entorno y Claves API

In [5]:
import os
import getpass

# --- Cargar Credenciales (Google Colab Secrets o .env Local) ---
try:
    from google.colab import userdata          # Entra aquí si estás en Google Colab

    # Lista de variables a buscar en los Secrets de Colab
    colab_keys = (
        "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL",
        "EMBEDDING_API_KEY", "EMBEDDING_BASE_URL", "EMBEDDING_MODEL"
    )
    for key in colab_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass                                # Si el Secret no existe en Colab, no hace nada
except ImportError:
    from dotenv import load_dotenv             # Entra aquí si estás corriendo en tu máquina local
    load_dotenv()

# --- Configuración por defecto si no están definidas en entorno/secrets ---
os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", "groq/compound-mini")

os.environ.setdefault("EMBEDDING_BASE_URL", "https://api.mistral.ai/v1")
os.environ.setdefault("EMBEDDING_MODEL", "mistral-embed")

# --- Asignación de Variables ---
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
# Busca en Secrets/env; si no existe, te pedirá ingresarla interactivamente como respaldo
LLM_API_KEY = os.getenv("LLM_API_KEY") or getpass.getpass("Ingresa tu LLM_API_KEY (Groq): ")

EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY") or getpass.getpass("Ingresa tu EMBEDDING_API_KEY (Mistral): ")

# --- Configuración del Proyecto ---
from pathlib import Path

def localizar_data(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if (candidato / "notebook" / "asistente_biblioteca_duoc.ipynb").is_file():
            carpeta = candidato / "data"
            if not carpeta.is_dir():
                raise FileNotFoundError(f"Falta la carpeta de datos del proyecto: {carpeta}")
            return carpeta
    raise FileNotFoundError(
        "Abre el notebook desde la carpeta del proyecto o desde su subcarpeta notebook. "
        "En Colab, carga el proyecto completo conservando data/ y notebook/."
    )

DATA_DIR = str(localizar_data())
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
TOP_K = 3
CONFIDENCE_THRESHOLD = 0.45

print("Configuración cargada correctamente:")
print(f"- LLM Model: {LLM_MODEL} (Groq)")
print(f"- Embedding Model: {EMBEDDING_MODEL} (Mistral)")

Configuración cargada correctamente:
- LLM Model: groq/compound-mini (Groq)
- Embedding Model: mistral-embed (Mistral)


## 3. Validación de los documentos existentes

Se utiliza únicamente `data/` en la raíz del proyecto, al mismo nivel que `notebook/`. La ruta se resuelve tanto al ejecutar desde la raíz como desde `notebook/`. No se crean ni se sobrescriben documentos. Todos las reglas de la biblioteca se encuentran dentro de la carpeta llamada data.


In [6]:
# Comprobar los documentos de la carpeta data/ del proyecto sin modificarlos.
ARCHIVOS_REQUERIDOS = (
    "reglamento_general.txt", "prestamos_renovaciones.txt", "morosos_sanciones.txt",
    "normas_disciplinarias.txt", "salas_estudio.txt", "uso_lentes_vr.txt",
)
faltantes = [nombre for nombre in ARCHIVOS_REQUERIDOS if not (Path(DATA_DIR) / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan documentos en {DATA_DIR}: {', '.join(faltantes)}")
print(f"Se leerán {len(ARCHIVOS_REQUERIDOS)} documentos existentes desde: {DATA_DIR}")


Se leerán 6 documentos existentes desde: C:\Users\AB\AppData\Local\GitHubDesktop\app-3.5.0\Rag_Reglamento_Biblioteca\data


## 4. Ingesta: Carga, Fragmentación y Construcción del Índice Vectorial (FAISS)

In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

DOCUMENT_TYPES = {
    "reglamento_general.txt": "reglamento_general",
    "prestamos_renovaciones.txt": "prestamos",
    "morosos_sanciones.txt": "sanciones",
    "normas_disciplinarias.txt": "normas_disciplinarias",
    "salas_estudio.txt": "salas_estudio",
    "uso_lentes_vr.txt": "lentes_vr",
}

def load_documents(data_dir: str = DATA_DIR) -> list[Document]:
    documents = []
    for filename, doc_type in DOCUMENT_TYPES.items():
        path = os.path.join(data_dir, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"No se encontró el documento requerido: {path}")
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append(
            Document(page_content=text, metadata={"tipo": doc_type, "fuente": filename})
        )
    return documents

def split_documents(documents: list[Document], chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\nArtículo", "\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documents)

def build_embeddings():
    return OpenAIEmbeddings(
        base_url=EMBEDDING_BASE_URL,
        api_key=EMBEDDING_API_KEY,
        model=EMBEDDING_MODEL,
        check_embedding_ctx_length=False,
    )

# --- Ejecutar la ingesta ---
docs = load_documents()
chunks = split_documents(docs)
embeddings = build_embeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Índice FAISS construido exitosamente con {vectorstore.index.ntotal} chunks.")

C:\Users\AB\AppData\Local\Temp\ipykernel_32720\2986993900.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Índice FAISS construido exitosamente con 36 chunks.


## 5. Formulación de Prompts (Prompts Engineering)

In [8]:
SYSTEM_INSTRUCTIONS = """Eres el Asistente Virtual de la Biblioteca Duoc UC (sede Plaza Norte).
Respondes exclusivamente con base en el CONTEXTO entregado, que proviene del
Reglamento de Bibliotecas Duoc UC y de información operativa de la sede.

Reglas estrictas:
1. No inventes plazos, montos de multas, artículos ni condiciones que no estén explícitos en el CONTEXTO.
2. Si el CONTEXTO no contiene información suficiente para responder con seguridad, indica que no cuentas con esa información y que se dirija con su consulta a un encargado de biblioteca. No intentes adivinar.
3. Cuando cites una norma, menciona el número de artículo si aparece en el CONTEXTO (ej. "según el Artículo 15°...").
4. Responde en español, de forma breve, clara y cordial.
"""

DERIVATION_MESSAGE = (
    "No tengo información suficiente y verificada en el reglamento para responder "
    "esa consulta con seguridad. Por favor, dirígete a un encargado de "
    "Biblioteca Duoc UC para que te ayude directamente."
)

FEW_SHOT_EXAMPLES = [
        {
        "pregunta": (
            "Soy alumno regular, ¿cuántos días dura el préstamo "
            "de un libro de la Colección General y bajo qué "
            "condiciones puedo renovarlo?"
        ),
        "contexto": (
            "Artículo 15°\n"
            "Colección General: tiempo de préstamo de 7 días, "
            "con un límite de 10 renovaciones por semestre.\n"
            "La renovación solo es aplicable cuando el material "
            "no se encuentra reservado y no se haya sobrepasado "
            "el límite de renovaciones permitidas."
        ),
        "respuesta": (
            "Según el Artículo 15°, el préstamo de un libro de la "
            "Colección General dura 7 días. El límite es de "
            "10 renovaciones por semestre, siempre que el material "
            "no esté reservado y no hayas alcanzado ese límite."
        ),
    },
    {
        "pregunta": "¿Puedo pedir un descuento en la multa si soy buen alumno?",
        "contexto": "El usuario que se encuentre en mora se hará acreedor de una multa de $1.000 por cada 7 días de atraso, por ítem.",
        "respuesta": DERIVATION_MESSAGE,
    },
]

def build_prompt(context: str, question: str, technique: str = "zero-shot") -> str:
    if technique == "zero-shot":
        return f"{SYSTEM_INSTRUCTIONS}\n\nCONTEXTO:\n{context}\n\nPREGUNTA DEL ESTUDIANTE:\n{question}\n\nRESPUESTA:"
    elif technique == "few-shot":
        ejemplos = "\n\n".join(
            f"Pregunta: {ej['pregunta']}\nContexto: {ej['contexto']}\nRespuesta: {ej['respuesta']}"
            for ej in FEW_SHOT_EXAMPLES
        )
        return f"{SYSTEM_INSTRUCTIONS}\n\nEjemplos:\n{ejemplos}\n\nNUEVO CASO:\nCONTEXTO:\n{context}\n\nPREGUNTA:\n{question}\n\nRESPUESTA:"
    elif technique == "chain-of-thought":
        return f"{SYSTEM_INSTRUCTIONS}\n\nCONTEXTO:\n{context}\n\nPREGUNTA:\n{question}\n\nRazona paso a paso qué artículos aplican y entrega SOLO la respuesta final al estudiante.\n\nRESPUESTA:"
    else:
        raise ValueError(f"Técnica desconocida: {technique}")

## 6. Agente RAG y filtro de recuperación

### Modelos de respaldo ante límites de Groq

El agente prueba primero el modelo configurado (`groq/compound-mini` actualmente). Si Groq devuelve **429 (límite de uso)**, prueba `openai/gpt-oss-20b` y después `openai/gpt-oss-120b`, usando la misma clave de Groq y el mismo contexto. Mistral continúa generando los embeddings.

Esto puede ayudar si otro modelo tiene cupo disponible, pero no garantiza superar límites compartidos de la organización o de los modelos internos de Compound. Si todos fallan, se informa que hay que esperar. Los errores distintos de 429 se mantienen visibles.

`agent.last_model` indica qué modelo respondió y `agent.last_attempts` registra los intentos sin credenciales. Para una futura comparación de técnicas de prompting, se puede utilizar fallback_models=[] al crear el agente para mantener el mismo modelo durante todas las pruebas. Si se alcanza su límite de uso, será necesario esperar o repetir toda la comparación con otro modelo fijo.

In [9]:
import time
from openai import OpenAI, RateLimitError

def _score_from_distance(distance: float) -> float:
    return 1.0 / (1.0 + max(distance, 0.0))

MODELOS_RESPALDO = ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]

class BibliotecaDuocAgent:
    def __init__(self, vectorstore, llm_client=None, model=None, threshold=None, top_k=None, fallback_models=None):
        self.vectorstore = vectorstore
        self.llm_client = llm_client or OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, max_retries=0, timeout=60)
        self.model = model or LLM_MODEL
        respaldos = MODELOS_RESPALDO if fallback_models is None else fallback_models
        self.models = list(dict.fromkeys([self.model, *respaldos]))
        self.last_model = None
        self.last_attempts = []
        self.threshold = CONFIDENCE_THRESHOLD if threshold is None else threshold
        self.top_k = TOP_K if top_k is None else top_k

    def retrieve(self, question: str):
        results = self.vectorstore.similarity_search_with_score(question, k=self.top_k)
        return [(doc, _score_from_distance(distance)) for doc, distance in results]

    @staticmethod
    def _build_context(results) -> str:
        return "\n\n".join(
            f"[Artículo/fuente: {doc.metadata.get('fuente', 'desconocida')}]\n{doc.page_content}"
            for doc, _ in results
        )

    def answer(self, question: str, technique: str = "zero-shot") -> dict:
        self.last_model = None
        self.last_attempts = []
        results = self.retrieve(question)
        if not results:
            return {"respuesta": DERIVATION_MESSAGE, "score": 0.0, "fuentes": []}

        top_score = max(score for _, score in results)
        fuentes = [doc.metadata.get("fuente") for doc, _ in results]

        if top_score < self.threshold:
            return {"respuesta": DERIVATION_MESSAGE, "score": top_score, "fuentes": fuentes}

        context = self._build_context(results)
        prompt = build_prompt(context=context, question=question, technique=technique)

        completion = None
        for candidate in self.models:
            try:
                completion = self.llm_client.chat.completions.create(
                    model=candidate,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0,
                )
                self.last_model = candidate
                self.last_attempts.append({"modelo": candidate, "estado": "correcto"})
                break
            except RateLimitError as exc:
                # No reintentar a los 3 segundos un cupo que puede tardar minutos.
                # Solo los errores 429 activan el cambio; otros errores se propagan.
                espera = exc.response.headers.get("retry-after") if exc.response is not None else None
                self.last_attempts.append({"modelo": candidate, "estado": "limite_429", "retry_after": espera})
                continue

        if completion is None:
            raise RuntimeError(
                "Groq alcanzó el límite de uso en todos los modelos habilitados para esta consulta. "
                "Espera a que se restablezca el cupo. Puedes revisar agent.last_attempts "
                "para ver los modelos probados y el tiempo Retry-After, si Groq lo informó."
            ) from None

        respuesta = completion.choices[0].message.content.strip()
        return {"respuesta": respuesta, "score": top_score, "fuentes": fuentes}

# Instanciar el agente
agent = BibliotecaDuocAgent(vectorstore)

## 7. Ejemplo de Consulta Interactiva

In [10]:
pregunta_demo = "¿Cuántos días puedo mantener en préstamo un libro de la colección general?"
resultado = agent.answer(pregunta_demo)

print(f"Pregunta: {pregunta_demo}")
# Puntuación de recuperación; no indica la certeza de la respuesta.
print(f"Puntuación de recuperación: {resultado['score']:.2f}")
print(f"Fuentes: {resultado['fuentes']}")
print(f"\nRespuesta:\n{resultado['respuesta']}")
print(f"Modelo utilizado: {agent.last_model or 'No se llamó al LLM'}")


Pregunta: ¿Cuántos días puedo mantener en préstamo un libro de la colección general?
Puntuación de recuperación: 0.81
Fuentes: ['prestamos_renovaciones.txt', 'prestamos_renovaciones.txt', 'prestamos_renovaciones.txt']

Respuesta:
Según el Artículo 15° del Reglamento de Bibliotecas Duoc UC, el tiempo de préstamo para la **Colección General** es de **7 días**.
Modelo utilizado: groq/compound-mini


## Pregunta interactiva Fuera de alcance

In [11]:
pregunta_demo2 = "¿Como puedo instalar linux en una computadora?"
resultado2 = agent.answer(pregunta_demo2)

print(f"\nPregunta: {pregunta_demo2}")
# Puntuación de recuperación; no indica la certeza de la respuesta.
print(f"Puntuación de recuperación: {resultado2['score']:.2f}")
print(f"Fuentes: {resultado2['fuentes']}")
print(f"\nRespuesta:\n{resultado2['respuesta']}")
print(f"Modelo utilizado: {agent.last_model or 'No se llamó al LLM'}")

#Esta pregunta se hace probar el modelo y ver la fiabilidad de la respuesta, ya que no hay información suficiente en el reglamento para responderla con seguridad.



Pregunta: ¿Como puedo instalar linux en una computadora?
Puntuación de recuperación: 0.65
Fuentes: ['salas_estudio.txt', 'salas_estudio.txt', 'morosos_sanciones.txt']

Respuesta:
Lo siento, pero en el contexto proporcionado no se incluye información sobre la instalación de Linux en una computadora. Le recomiendo dirigirse a un encargado de la biblioteca para recibir la asistencia adecuada.
Modelo utilizado: groq/compound-mini


## 8. Evaluación con Casos Dentro y Fuera de Alcance

In [13]:
CASOS_DENTRO_DE_ALCANCE = [
    "¿Puedo reservar la sala de estudio para una fiesta de cumpleaños?",
    "¿Cuántos días puedo mantener en préstamo un libro de la colección general?",
    "¿Cuánto es la multa por atraso en la devolución de un libro?",
    "¿Cuántas horas al día puedo reservar una sala de estudio?",
    "Perdí el libro que tenía prestado, ¿qué debo hacer?",
    "¿Puedo comer dentro de las salas de estudio?",
]

CASOS_FUERA_DE_ALCANCE = [
    "¿Pueden hacerme un descuento especial en la multa porque soy buen alumno?",
    "¿La biblioteca vende café o snacks?",
]

def run_case(agent, question: str, expectativa: str):
    resultado = agent.answer(question)
    print(f"Modelo utilizado: {agent.last_model or 'No se llamó al LLM'}")
    time.sleep(1)
    print(f"Pregunta: {question}")
    print(f"Criterio de revisión: {expectativa}")
    print(f"Score: {resultado['score']:.2f}")
    print(f"Fuentes: {resultado['fuentes']}")
    print(f"Respuesta: {resultado['respuesta']}\n")
    return resultado

print("Evaluación manual: contrastar cada respuesta con las fuentes y el criterio indicado.\n")
res_dentro = [run_case(agent, q, "Responder con información respaldada y citar el artículo aplicable.")
              for q in CASOS_DENTRO_DE_ALCANCE]
res_fuera = [run_case(agent, q, "Reconocer la información faltante e indicar que contacte a un encargado.")
             for q in CASOS_FUERA_DE_ALCANCE]
print(f"Casos ejecutados: {len(res_dentro) + len(res_fuera)}.")


Evaluación manual: contrastar cada respuesta con las fuentes y el criterio indicado.

Modelo utilizado: groq/compound-mini
Pregunta: ¿Puedo reservar la sala de estudio para una fiesta de cumpleaños?
Criterio de revisión: Responder con información respaldada y citar el artículo aplicable.
Score: 0.75
Fuentes: ['salas_estudio.txt', 'salas_estudio.txt', 'salas_estudio.txt']
Respuesta: No. Según el Artículo 27° del Reglamento de Bibliotecas, las salas de estudio están destinadas exclusivamente a actividades académicas y está prohibido utilizarlas para actividades no académicas, como una fiesta de cumpleaños. Por lo tanto, no es posible reservar la sala para ese fin.

Modelo utilizado: groq/compound-mini
Pregunta: ¿Cuántos días puedo mantener en préstamo un libro de la colección general?
Criterio de revisión: Responder con información respaldada y citar el artículo aplicable.
Score: 0.81
Fuentes: ['prestamos_renovaciones.txt', 'prestamos_renovaciones.txt', 'prestamos_renovaciones.txt']
Resp

## Revisión manual de los resultados

Se contrastaron las ocho respuestas con el reglamento incluido en `data/`.

- **Modelo utilizado:** `groq/compound-mini` en las ocho consultas.
- **Técnica:** zero-shot.
- **Umbral de recuperación:** 0.45.
- **Criterios:** exactitud de la información, correspondencia de las citas y reconocimiento de información no disponible.
- **Interpretación del score:** puntuación de recuperación; no representa un porcentaje de certeza.

| N.º | Consulta | Respuesta esperada | Referencia | Resultado | Observaciones |
|---|---|---|---|---|---|
| 1 | Reservar una sala para un cumpleaños | Indicar que no está permitido porque las salas están destinadas a actividades académicas. | Artículo 27 | Cumple | Rechaza el uso solicitado y cita correctamente la norma. Score: 0.75. |
| 2 | Plazo de préstamo de Colección General | Indicar un plazo de 7 días para el contexto de un estudiante de Duoc UC. | Artículo 15 | Cumple | Informa el plazo correcto y cita el artículo correspondiente. Esta prueba no evalúa las condiciones particulares de usuarios externos. Score: 0.81. |
| 3 | Multa por atraso | Indicar $1.000 por cada 7 días de atraso, por ítem, aplicable únicamente a usuarios internos. | Artículo 18 | Cumple | Incluye el monto, la periodicidad, la aplicación por ítem y la restricción a usuarios internos. Score: 0.79. |
| 4 | Tiempo diario de reserva de sala | Indicar un máximo de una hora diaria por usuario. | Artículo 26 | Cumple | Responde correctamente y cita el artículo aplicable. Score: 0.77. |
| 5 | Pérdida de un libro | Indicar reposición en 20 días hábiles y que el encargado define un equivalente de igual valor si el material no está disponible. | Artículo 22; artículos 17 y 13 para las condiciones de mora y sus efectos | Cumple con observación | La reposición y su excepción están correctamente explicadas. La recomendación adicional sobre mora debe distinguirse de la obligación de reposición: el artículo 13 establece la restricción cuando el usuario está en mora, no que la pérdida produzca automáticamente ese estado. Score: 0.71. |
| 6 | Comer dentro de las salas | Indicar que no se permiten alimentos y que solo se autorizan líquidos en recipientes no propensos a derrames. | Artículo 27 | Cumple | Explica correctamente la prohibición y la excepción sobre bebidas. Score: 0.75. |
| 7 | Descuento por buen rendimiento académico | Indicar que no se dispone de información que autorice ese descuento y recomendar consultar a un encargado. | Artículo 18 como referencia de la multa; sin disposición sobre descuentos en los documentos consultados | Cumple con observación | No inventa descuentos y recomienda atención humana. Es más preciso decir “no encontré información sobre descuentos en el reglamento consultado” para evitar interpretar la ausencia de información como una prohibición expresa. Score: 0.73. |
| 8 | Venta de café o snacks | Reconocer que la documentación no contiene esa información y recomendar consultar a biblioteca. | Sin información en los documentos consultados | Cumple | Reconoce el límite documental sin inventar una respuesta. Score: 0.71. |

### Alcance de esta revisión

Las ocho respuestas atienden el criterio principal esperado. Se identificaron dos observaciones de precisión en los casos 5 y 7.

Los resultados corresponden a esta ejecución y a estas ocho preguntas. No demuestran exactitud para todas las consultas posibles ni constituyen una comparación entre técnicas de prompting.

Todas las consultas superaron el umbral de 0.45. Por ello, esta ejecución muestra el reconocimiento de información insuficiente mediante el modelo, pero no prueba la respuesta automática del filtro para puntuaciones inferiores al umbral.